# BGE Text Embeddings for Beauty and Personal Care Metadata

This notebook preprocesses `features` and `description` from `meta_Beauty_and_Personal_Care.jsonl`, embeds the cleaned product text with `BAAI/bge-base-en-v1.5`, and stores chunked Parquet embedding files keyed by `parent_asin`.

Default behavior is a small test run. Change `MAX_ROWS = None` in the config cell when you are ready for the full file.

In [2]:
# Install dependencies in the current notebook kernel.
# Run this once if your environment does not already have these packages.
%pip install -q torch sentence-transformers pandas pyarrow tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
from __future__ import annotations

import html
import json
import math
import re
import unicodedata
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

/opt/homebrew/Caskroom/miniconda/base/envs/rcd_proj01/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ----------------------------
# Config
# ----------------------------

PROJECT_DIR = Path("/Users/frankwang1224/Projects/rcd_sys_proj02")
INPUT_PATH = PROJECT_DIR / "dataset/meta_Beauty_and_Personal_Care.jsonl"
OUTPUT_DIR = PROJECT_DIR / "embeddings/text_bge_base_en_v1_5"

MODEL_NAME = "BAAI/bge-base-en-v1.5"
CATEGORY = "Beauty_and_Personal_Care"

# Use None for automatic choice. On Apple Silicon, you can try DEVICE = "mps".
DEVICE = None

# Safety setting: start small. Change to None for the full file.
MAX_ROWS = None

BATCH_SIZE = 64
CHUNK_SIZE = 10_000

MAX_FEATURE_BULLETS = 8
MAX_DESCRIPTION_CHARS = 1_000
MAX_TOTAL_CHARS = 2_000

print(f"Input:      {INPUT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model:      {MODEL_NAME}")
print(f"MAX_ROWS:   {MAX_ROWS}")

Input:      /Users/frankwang1224/Projects/rcd_sys_proj02/dataset/meta_Beauty_and_Personal_Care.jsonl
Output dir: /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5
Model:      BAAI/bge-base-en-v1.5
MAX_ROWS:   None


## Preprocessing Plan

Accepted plan:

1. Remove emojis/fancy Unicode.
2. Remove refund/contact/shipping/service sentences.
3. Truncate long text.
4. Keep product features and description.
5. Use `parent_asin` as the item id.

In [5]:
TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|;\s+|\n+")

SERVICE_NOISE_RE = re.compile(
    r"\b("
    r"refund|return|replacement|warranty|guarantee|risk free|"
    r"no questions asked|customer service|after[- ]?sales|"
    r"contact us|contact seller|feel free to contact|"
    r"shipping|delivery|amazon fba|fba|prime|"
    r"satisfaction guaranteed|money back"
    r")\b",
    re.IGNORECASE,
)


def normalize_unicode(text: str) -> str:
    """Normalize fancy Unicode letters and remove emoji/decorative symbols."""
    text = unicodedata.normalize("NFKC", text)
    kept = []

    for char in text:
        category = unicodedata.category(char)

        # Drop emoji/decorative symbols and control chars.
        if category in {"So", "Sk", "Cc", "Cf"}:
            kept.append(" ")
            continue

        kept.append(char)

    return "".join(kept)


def clean_text(value: object) -> str:
    if value is None:
        return ""

    text = str(value)
    text = html.unescape(text)
    text = TAG_RE.sub(" ", text)
    text = normalize_unicode(text)
    text = SPACE_RE.sub(" ", text)
    return text.strip()


def value_to_clean_list(value: object) -> list[str]:
    """Convert a JSON field into a de-duplicated list of clean strings."""
    if value is None:
        return []

    values = value if isinstance(value, list) else [value]
    cleaned = []
    seen = set()

    for raw in values:
        text = clean_text(raw)
        if not text:
            continue

        key = text.casefold()
        if key in seen:
            continue

        seen.add(key)
        cleaned.append(text)

    return cleaned


def remove_service_noise(text: str) -> str:
    """Remove sentences/bullets about refund, shipping, warranty, contact, etc."""
    pieces = [p.strip() for p in SENTENCE_SPLIT_RE.split(text) if p.strip()]
    kept = [p for p in pieces if not SERVICE_NOISE_RE.search(p)]
    return SPACE_RE.sub(" ", "; ".join(kept)).strip()


def truncate_on_word(text: str, max_chars: int) -> str:
    if max_chars <= 0 or len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0].strip()


def build_embedding_text(
    features: list[str],
    description: list[str],
    max_feature_bullets: int,
    max_description_chars: int,
    max_total_chars: int,
) -> str:
    clean_features = []
    for item in features[:max_feature_bullets]:
        item = remove_service_noise(item)
        if item:
            clean_features.append(item)

    description_text = remove_service_noise(" ".join(description))
    description_text = truncate_on_word(description_text, max_description_chars)

    parts = []
    if clean_features:
        parts.append("Features: " + "; ".join(clean_features))
    if description_text:
        parts.append("Description: " + description_text)

    text = SPACE_RE.sub(" ", " ".join(parts)).strip()
    return truncate_on_word(text, max_total_chars)

In [7]:
def iter_preprocessed_rows(
    input_path: Path,
    max_rows: int | None,
    max_feature_bullets: int,
    max_description_chars: int,
    max_total_chars: int,
) -> Iterable[dict[str, str]]:
    scanned = 0

    with input_path.open("r", encoding="utf-8") as file:
        for line in file:
            if max_rows is not None and scanned >= max_rows:
                break

            scanned += 1

            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            parent_asin = row.get("parent_asin")
            if not parent_asin:
                continue

            features = value_to_clean_list(row.get("features"))
            description = value_to_clean_list(row.get("description"))

            embedding_text = build_embedding_text(
                features=features,
                description=description,
                max_feature_bullets=max_feature_bullets,
                max_description_chars=max_description_chars,
                max_total_chars=max_total_chars,
            )

            if not embedding_text:
                continue

            yield {
                "parent_asin": str(parent_asin),
                "embedding_text": embedding_text,
            }


def write_chunk(records: list[dict[str, object]], output_dir: Path, category: str, chunk_index: int) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"text_embeddings_{category}_{chunk_index:05d}.parquet"
    pd.DataFrame(records).to_parquet(output_path, index=False)
    return output_path

In [8]:
# Preview the cleaned embedding text before loading the model.
preview_rows = list(
    iter_preprocessed_rows(
        input_path=INPUT_PATH,
        max_rows=20,
        max_feature_bullets=MAX_FEATURE_BULLETS,
        max_description_chars=MAX_DESCRIPTION_CHARS,
        max_total_chars=MAX_TOTAL_CHARS,
    )
)

for row in preview_rows[:3]:
    print("parent_asin:", row["parent_asin"])
    print(row["embedding_text"][:800])
    print("-" * 80)

parent_asin: B08BLDKYHB
Features: [MEET YOUR HAIR COLOR NEEDS] Bright color, high-quality hair chalk.; The comb applicators make it quick and easy to put dazzling colors into your hair.; Your hair will be supple and natural, not stiff and sticky.; [Shiyeen COLORFUL HAIR CHALK COMBS SET] Contains White, purple, blue, orange, pink, red, cyan, green, brown, Rose ,10 bright colors, Disposable gloves, hair dye shawl.; [NON-TOXIC & EASY TO WASH] The hair chalk is non-allergenic , non-toxic and water-soluble and environmentally friendly; apply it to hair evenly from top to bottom and washes out with ordinary shampoo and water.; [USING SIMPLE] The hair chalks are perfect for going wild!; Even kids are able to use them on their own super easily.; With a waxy lipstick texture, DIY your hair color with your different dress-
--------------------------------------------------------------------------------
parent_asin: B0BWJGQ32Y
Features: Frontal Wigs Human Hair Material: 100% unprocessed Brazilian

In [9]:
# Load BGE model. First run will download the model.
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 512
print(model)
print("max_seq_length:", model.max_seq_length)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8796.33it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)
max_seq_length: 512


In [10]:
def run_embedding() -> dict:
    pending_texts = []
    pending_ids = []
    output_records = []
    written_files = []
    chunk_index = 0
    embedded_count = 0

    rows = iter_preprocessed_rows(
        input_path=INPUT_PATH,
        max_rows=MAX_ROWS,
        max_feature_bullets=MAX_FEATURE_BULLETS,
        max_description_chars=MAX_DESCRIPTION_CHARS,
        max_total_chars=MAX_TOTAL_CHARS,
    )

    progress_total = MAX_ROWS if MAX_ROWS is not None else None
    progress = tqdm(rows, total=progress_total, desc="preprocess/embed")

    def flush_batch():
        nonlocal pending_texts, pending_ids, output_records, chunk_index
        nonlocal embedded_count, written_files

        if not pending_texts:
            return

        embeddings = model.encode(
            pending_texts,
            batch_size=BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        for parent_asin, text, emb in zip(pending_ids, pending_texts, embeddings):
            output_records.append(
                {
                    "parent_asin": parent_asin,
                    "text_embedding": emb.tolist(),
                    "embedding_text": text,
                    "embedding_model": MODEL_NAME,
                    "embedding_dim": int(emb.shape[0]),
                }
            )

        embedded_count += len(pending_texts)
        pending_texts = []
        pending_ids = []

        if len(output_records) >= CHUNK_SIZE:
            path = write_chunk(output_records, OUTPUT_DIR, CATEGORY, chunk_index)
            written_files.append(path)
            print(f"Wrote {path} ({len(output_records):,} rows)")
            output_records = []
            chunk_index += 1

    for row in progress:
        pending_ids.append(row["parent_asin"])
        pending_texts.append(row["embedding_text"])

        if len(pending_texts) >= BATCH_SIZE:
            flush_batch()
            progress.set_postfix(embedded=f"{embedded_count:,}")

    flush_batch()

    if output_records:
        path = write_chunk(output_records, OUTPUT_DIR, CATEGORY, chunk_index)
        written_files.append(path)
        print(f"Wrote {path} ({len(output_records):,} rows)")

    manifest = {
        "input": str(INPUT_PATH),
        "model": MODEL_NAME,
        "category": CATEGORY,
        "embedding_dim": 768,
        "normalized_embeddings": True,
        "max_seq_length": model.max_seq_length,
        "max_feature_bullets": MAX_FEATURE_BULLETS,
        "max_description_chars": MAX_DESCRIPTION_CHARS,
        "max_total_chars": MAX_TOTAL_CHARS,
        "rows_embedded": embedded_count,
        "num_output_files": len(written_files),
        "output_files": [str(path) for path in written_files],
    }

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = OUTPUT_DIR / f"text_embeddings_{CATEGORY}_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print("\nDone.")
    print(f"Rows embedded: {embedded_count:,}")
    print(f"Output files:  {len(written_files):,}")
    print(f"Manifest:      {manifest_path}")

    if embedded_count:
        num_chunks = math.ceil(embedded_count / CHUNK_SIZE)
        print(f"Expected chunk count from row total: about {num_chunks:,}")

    return manifest

In [11]:
# Run embedding.
# With MAX_ROWS = 1_000, this is a safe test run.
# For the full file, change MAX_ROWS = None in the config cell and rerun from there.
manifest = run_embedding()

preprocess/embed: 10048it [03:47, 40.82it/s, embedded=10,048]

Wrote /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00000.parquet (10,048 rows)


preprocess/embed: 20096it [07:42, 40.84it/s, embedded=20,096]

Wrote /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00001.parquet (10,048 rows)


preprocess/embed: 30144it [11:49, 39.58it/s, embedded=30,144]

...

Wrote /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00077.parquet (10,048 rows)


preprocess/embed: 793792it [5:28:05, 37.20it/s, embedded=793,792]

Wrote /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00078.parquet (10,048 rows)


preprocess/embed: 803838it [5:32:15, 40.32it/s, embedded=803,776]


Wrote /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00079.parquet (10,046 rows)

Done.
Rows embedded: 803,838
Output files:  80
Manifest:      /Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_manifest.json
Expected chunk count from row total: about 81


In [12]:
# Inspect one output file.
output_files = sorted(OUTPUT_DIR.glob(f"text_embeddings_{CATEGORY}_*.parquet"))
output_files[:3], len(output_files)

([PosixPath('/Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00000.parquet'),
  PosixPath('/Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00001.parquet'),
  PosixPath('/Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/text_bge_base_en_v1_5/text_embeddings_Beauty_and_Personal_Care_00002.parquet')],
 80)

In [13]:
if output_files:
    sample_df = pd.read_parquet(output_files[0])
    display(sample_df.head())
    print("embedding length:", len(sample_df.loc[0, "text_embedding"]))

,parent_asin,text_embedding,embedding_text,embedding_model,embedding_dim
0,B08BLDKYHB,"[-0.056411802768707275, -0.0166774969547987, -...",Features: [MEET YOUR HAIR COLOR NEEDS] Bright ...,BAAI/bge-base-en-v1.5,768
1,B0BWJGQ32Y,"[0.014209776185452938, 0.007619387935847044, 0...",Features: Frontal Wigs Human Hair Material: 10...,BAAI/bge-base-en-v1.5,768
2,B0BM8WLSXF,"[-0.028382442891597748, 0.007068037986755371, ...",Features: 3 Inch Clipper Guards: The only 3 in...,BAAI/bge-base-en-v1.5,768
3,B00N4LMZZK,"[-0.057587672024965286, -0.03493224084377289, ...",Features: Magic Cream is formulated with an ad...,BAAI/bge-base-en-v1.5,768
4,B01DX1OEFO,"[-0.018158884719014168, -0.00872894935309887, ...",Features: Intense color; Matte finish; Silky t...,BAAI/bge-base-en-v1.5,768


embedding length: 768
